# Import

In [1]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

In [2]:
import torch
import numpy as np
import mmap

In [3]:
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torchvision import transforms
import concurrent.futures
from PIL import Image
import shutil

In [4]:
import albumentations as albu


In [5]:
from torch.utils.data import Dataset
from typing import Union,List,Tuple
import pathlib
from timeit import default_timer as timer
from torch.utils.data import DataLoader
import h5py
from tqdm import tqdm
import torchvision
import matplotlib.pyplot as plt
from sklearn.utils import shuffle
import time
from torch.utils.tensorboard import SummaryWriter

In [6]:
# mlfmow library
import mlflow
import mlflow.pytorch


# MLflow run

In [7]:
from mlflow.exceptions import MlflowException
import datetime

# End MLflow run
mlflow.end_run()

# Function to generate a unique experiment name
def generate_experiment_name():
    return f"experiment-{datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}"

# Function to create and set a new experiment
def set_experiment(experiment_name):
    try:
        experiment = mlflow.get_experiment_by_name(experiment_name)
        if experiment is None:
            mlflow.create_experiment(experiment_name)
            experiment = mlflow.get_experiment_by_name(experiment_name)
        mlflow.set_experiment(experiment_name)
    except MlflowException as e:
        print(f"Error setting experiment: {e}")
        
        
# Generate a unique experiment name and set the experiment
experiment_name = generate_experiment_name()
set_experiment(experiment_name)

# Verify the experiment is set correctly
print(f"Set experiment '{experiment_name}'")


# Start an MLflow run
mlflow.start_run(run_name=f"training_DEB-{datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Unique runs directory
runs_dir = os.path.join("runs", experiment_name, datetime.datetime.now().strftime('%Y-%m-%d_%H-%M-%S'))



Set experiment 'experiment-2024-07-09_16-46-51'


In [8]:
#Put the device on GPU if possible to train the model faster than with CPU

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Log device information
mlflow.log_param("device", "cuda:0" if torch.cuda.is_available() else "cpu")

batch_size = 32
period_size = 8
weight_decay = 0.1
epsilon = 10**(-8)
epochs_num = 10
learning_rate = 10**(-3)
n_channel = 1
class_num = 3
dropout_rate = 0.5

# Define your source and destination directories
label_dir = "D:/Donnees_pour_segmentation_RDN/data/ROU_2006/Label"
input_dir = "D:/Donnees_pour_segmentation_RDN/data/ROU_2006/Unseg"
new_label_dir = "D:/Donnees_pour_segmentation_RDN/data/ROU_2006_mini/Label"
new_input_dir = "D:/Donnees_pour_segmentation_RDN/data/ROU_2006_mini/Unseg"

runs_dir = os.path.join("runs", datetime.datetime.now().strftime('%Y-%m-%d%H-%M-%S'))

stride = 32
train_size = 0.7

output = 256

# Log parameters
mlflow.log_param("batch_size", batch_size)
mlflow.log_param("period_size", period_size)
mlflow.log_param("weight_decay", weight_decay)
mlflow.log_param("epsilon", epsilon)
mlflow.log_param("epochs_num", epochs_num)
mlflow.log_param("learning_rate", learning_rate)
mlflow.log_param("n_channel", n_channel)
mlflow.log_param("class_num", class_num)
mlflow.log_param("label_dir", label_dir)
mlflow.log_param("input_dir", input_dir)
mlflow.log_param("runs_dir", runs_dir)
mlflow.log_param("stride", stride)
mlflow.log_param("train_size", train_size)
mlflow.log_param("output", output)
mlflow.log_param("dropout_rate", dropout_rate)

# Initialize TensorBoard writer
writer = SummaryWriter(runs_dir)

# Select files

In [9]:
# Ensure the new directories exist
os.makedirs(new_label_dir, exist_ok=True)
os.makedirs(new_input_dir, exist_ok=True)

# Function to empty the destination directories
def empty_directory(directory):
    for filename in os.listdir(directory):
        file_path = os.path.join(directory, filename)
        if os.path.isfile(file_path) or os.path.islink(file_path):
            os.unlink(file_path)
        elif os.path.isdir(file_path):
            shutil.rmtree(file_path)

# Function to copy files
def copy_files(file_paths, destination_dir):
    for file_path in file_paths:
        shutil.copy(file_path, destination_dir)

# Function to get the number of digits in the file names
def get_num_digits(directory):
    files = [f for f in os.listdir(directory) if f.endswith('.tif')]
    if not files:
        raise ValueError("No .tif files found in the directory")
    num_digits = len(files[0].split('_')[-1].split('.')[0])
    return num_digits

# Function to get image files by index range
def get_image_files_by_index(directory, start_index, end_index):
    num_digits = get_num_digits(directory)
    image_files = []
    for idx in range(start_index, end_index + 1):
        file_name = f"{directory}/ROU_2006_{idx:0{num_digits}}.tif"
        if os.path.exists(file_name):
            image_files.append(file_name)
        else:
            print(f"File not found: {file_name}")
    return image_files

# Empty the destination directories
empty_directory(new_label_dir)
empty_directory(new_input_dir)

# Get the list of image files
label_files = get_image_files_by_index(label_dir, start_index=800, end_index=850)
input_files = get_image_files_by_index(input_dir, start_index=800, end_index=850)

# Copy the files to the new directories
copy_files(label_files, new_label_dir)
copy_files(input_files, new_input_dir)

print("Files copied successfully!")


Files copied successfully!


# Dataset

In [10]:
def load_patches(patches):

    if isinstance(patches, str):
        return np.array(pd.read_csv(patches, header=0)).tolist()
    else:
        return patches

class HDF52D(Dataset):

    # dataset for segmentation used
    def __init__(self, train_patches,val_patches, train_transform=None, val_transform=None, train_idx = None):

        self.patches = {'train': load_patches(train_patches),
                        'val': load_patches(val_patches)}

        self.transforms = {'train': train_transform,
                           'val': val_transform}

        self.train_idx = load_patches(train_idx)

        self.mode = 'train'

    def __getitem__(self, idx):

        p = self.patches[self.mode]
        
        image = input_list[p[0][idx]][int(p[1][idx]):int(p[1][idx]) + int(p[3][idx]), int(p[2][idx]):int(p[2][idx]) + int(p[4][idx]) ]
        mask = label_list[p[0][idx]][int(p[1][idx]):int(p[1][idx]) + int(p[3][idx]), int(p[2][idx]):int(p[2][idx]) + int(p[4][idx]) ]
        sample = {'image': image, 'mask': mask}

        if self.transforms[self.mode] is not None:
            sample = self.transforms[self.mode](sample)
            
        #if self.train_idx is not None and self.mode == 'train':
        #    sample['index'] = self.train_idx[idx]
        return sample


    def train(self):
        self.mode = 'train'

    def val(self):
        self.mode = 'val'

    def __len__(self):
        return len(self.patches[self.mode])

In [11]:
#print("MLflow tracking URI:", mlflow.get_tracking_uri())

# Dataprocess

In [12]:
def create_one_hot(mask, num_classes = 3):
    one_hot_mask = torch.zeros([mask.shape[0],
                                num_classes,
                                mask.shape[1],
                                mask.shape[2]],
                               dtype=torch.float32)
    if mask.is_cuda:
        one_hot_mask = one_hot_mask.cuda()
    one_hot_mask = one_hot_mask.scatter(1, mask.long().data.unsqueeze(1), 1.0)

    return one_hot_mask

def adjustMask(mask, class_num):

    interval = int(256.0 / class_num)

    # Color_Dict must be a numpy type
    # mask.shape must be a H x W x C
    # do not have channel dimensions
    if len(mask.shape) == 2:
        new_mask = np.zeros((mask.shape[0], mask.shape[1]), dtype=np.longlong)
        for i in range(class_num):
            if i <= class_num - 2:
                new_mask[(mask >= i*interval) & (mask < (i+1) * interval)] = i
            else:
                new_mask[i*interval <= mask] = i
        return new_mask

class AdjustMask(object):
    def __init__(self, class_num = 3):
        self.class_num = class_num

    def __call__(self, sample):
        sample['mask'] = adjustMask(sample['mask'], self.class_num)
        return sample

class ToTensor(object):

    def __init__(self, if_multi_img=False):
        self.if_multi_img = if_multi_img

    def __call__(self, sample):
        image, mask = sample['image'], sample['mask']

        # swap color axis because
        # numpy image: H x W x C
        # torch image: C x H x W

        if not self.if_multi_img:
            if len(image.shape) == 2:
                image = np.expand_dims(image, axis=2)
            image = image.transpose((2, 0, 1))
        else:
            if len(image.shape) == 3:
                image = np.expand_dims(image, axis=3)

            image = image.transpose((0, 3, 1, 2))

        sample['image'] = torch.from_numpy(image)
        sample['mask'] = torch.from_numpy(mask)

        if 'weights' in sample:
            sample['weights'] = torch.from_numpy(sample['weights'])
        if 'ratio' in sample:
            sample['ratio'] = torch.from_numpy(sample['ratio'])
        return sample

class Normalize(object):
    def __init__(self, max=255.0, min=0.0, tg_max=1.0, tg_min=0.0):
        self.max = max
        self.min = min
        self.tg_max = tg_max
        self.tg_min = tg_min

    def __call__(self, sample):
        image = sample['image'].astype('float32')
        image = self.tg_min + ((image - self.min)*(self.tg_max - self.tg_min)) / (self.max - self.min)
        sample['image'] = image
        return sample

class Augmentation(object):

    def __init__(self, output_size=256):
        self.aug = albu.Compose([
            albu.OneOf([
                albu.HorizontalFlip(p=1),
                albu.VerticalFlip(p=1),   
                albu.Compose([
                    albu.HorizontalFlip(p=1),
                    albu.VerticalFlip(p=1), 
                ])
            ], p=0.75),
            # albu.OneOf([
            # albu.RandomContrast(),
            # albu.RandomGamma(),
            # albu.RandomBrightness(),
            # ], p=0.5),
            # albu.OneOf([
            # albu.ElasticTransform(alpha=60, sigma=120 * 0.05, alpha_affine=120 * 0.03),
            # albu.GridDistortion(),
            # albu.OpticalDistortion(distort_limit=2, shift_limit=0.5),
            # ], p=0.),5
            albu.augmentations.geometric.rotate.RandomRotate90(p=1),
            albu.Resize(output_size, output_size, always_apply=True),
        ])
    def __call__(self,sample):
        augmented = self.aug(image=sample['image'], mask=sample['mask'])
        sample['image'] = augmented['image']
        sample['mask'] = augmented['mask']
        return sample

# Generate Patches

In [13]:
#get_minimum_dirt_patches create a subset of initial patches focusing on patches containing dirt (depending on dirt_rate)
# Input :
# dirt_choose_threshold: A threshold value for the dirt ratio to determine whether a patch is considered as a dirt patch.
# dirt_rate: The desired proportion of dirt patches in the selected subset.
# patches: A NumPy array containing information about patches (name, top, left, height, width).
# ratios: A NumPy array containing ratios for each patch (class ratios).
#
# Output :
# returns a subset of initial patches
def get_minimum_dirt_patches(dirt_choose_threshold: float, dirt_rate: float, patches: np.array, ratios: np.array):
    # Start a nested MLflow run for this function
    with mlflow.start_run(nested=True, run_name="get_minimum_dirt_patches"):
        # Log parameters
        mlflow.log_param("dirt_choose_threshold", dirt_choose_threshold)
        mlflow.log_param("dirt_rate", dirt_rate)
        
        # get ratios
        ratios = np.array(ratios)
        # get index that would sort ratios by decreasing order
        ratios_idx = np.argsort(-ratios, axis=0)

        # ratios dimension is n_patches x n_classes
        # get the second column of ratio_idx which is the dirt index sorted by decreasing order
        dirt_idx = ratios_idx[:, 1]

        # get only the patches that dirt ratio is > dirt_choose_threshold
        last_idx = 0
        for i in range(dirt_idx.shape[0]):
            dirt_ratio = ratios[dirt_idx[i], 1]
            if dirt_ratio < dirt_choose_threshold:
                last_idx = i
                break
        
        # indexes of wanted dirt_patches
        dirt_patches_idx = dirt_idx[0:last_idx]
        # indexes of other patches
        rest_idx = dirt_idx[last_idx:-1]

        if not (dirt_rate == 0):
            rest_num = round(((last_idx - 1) / dirt_rate) * (1 - dirt_rate))
            if rest_num > rest_idx.shape[0]:
                rest_num = rest_idx.shape[0]
        else:
            rest_num = rest_idx.shape[0]
        
        # Getting picking other patches
        random_idx = np.random.choice(rest_idx.shape[0], size=rest_num, replace=False)
        non_dirt_patches_idx = rest_idx[random_idx]

        # Getting the final patches
        patches_idx = np.concatenate((dirt_patches_idx, non_dirt_patches_idx), axis=0)
        new_patches = np.asarray(patches)[patches_idx, :].tolist()

        new_patches = shuffle(new_patches)
        new_patches = [[name, int(top), int(left), int(h), int(w)] for [name, top, left, h, w] in new_patches]
        
        # Log the number of patches generated
        mlflow.log_metric("num_dirt_patches", len(dirt_patches_idx))
        mlflow.log_metric("num_non_dirt_patches", len(non_dirt_patches_idx))
        
        return new_patches


#get_dirt_bone_patches create a subset of initial patches focusing on patches containing air (depending on air_rate)
# Input :
# patches: A NumPy array containing information about patches (name, top, left, height, width).
# ratios: A NumPy array containing ratios for each patch (class ratios).
# air_rate: The desired proportion of air patches in the selected subset.
#
# Output :
# returns a list of extracted patches based on the specified conditions and an index representing the length of this list
def get_dirt_bone_patches(patches: np.array, ratios: np.array, air_rate: float):
    # Start a nested MLflow run for this function
    with mlflow.start_run(nested=True, run_name="get_dirt_bone_patches"):
        # get ratios 
        ratios = np.array(ratios)
        # get indices that would sort ratios by decreasing order
        ratios_idx = np.argsort(-ratios, axis=0) 

        # get the second column of ratio_idx which is the dirt index sorted by decreasing order
        dirt_idx = ratios_idx[:, 1]
        # get patches 
        patches = np.asarray(patches)

        # get ratios and patches sorted by decreasing dirt ratios 
        ratios_sort = ratios[dirt_idx, :]
        patches_sort = patches[dirt_idx, :]
        
        dirt_patches = []
        bone_patches = []
        
        while (len(dirt_patches) < 128 or len(bone_patches) < 128) and air_rate < 1:  # in the case that we have not enough patches 
            dirt_patches = []
            bone_patches = []
            # get patches that contains significant differences between dirt and bone ratios (>0.1)
            for idx in range(ratios_sort.shape[0]):
                if (ratios_sort[idx, 0] < air_rate):
                    if (ratios_sort[idx, 1] - ratios_sort[idx, 2] > 0.15):
                        dirt_patches.append(patches_sort[idx, :].tolist())
                    elif (ratios_sort[idx, 2] - ratios_sort[idx, 1] > 0.15):
                        bone_patches.append(patches_sort[idx, :].tolist())
                else:
                    pass
            air_rate += 0.1

        dirt_len = len(dirt_patches)
        bone_len = len(bone_patches)

        bone_index = [1 for i in range(bone_len)]
        dirt_index = [0 for i in range(dirt_len)]
        
        dirt_patches = shuffle(dirt_patches)
        bone_patches = shuffle(bone_patches)

        print(f"There are {bone_len} bone and {dirt_len} dirt patches in the training data...")
        
        # Log the number of patches generated
        mlflow.log_metric("num_dirt_patches", dirt_len)
        mlflow.log_metric("num_bone_patches", bone_len)

        end_idx = dirt_len if dirt_len < bone_len else bone_len
        # get the same quantity of dirt and bone patches
        new_patches = []
        for patch in dirt_patches:
            new_patches.append(patch)
        for patch in bone_patches:
            new_patches.append(patch)

        d_index = len(new_patches)
        
        return new_patches, d_index


#slide_windows generate a list of patches based on a sliding window approach over an input image or data.
# Input :
# name : A string representing the name or identifier for the image or data.
# shape : A tuple representing the shape (height, width) of the input image or data.
# output_size (int): An integer or tuple representing the size of the output patches. Default is set to (128, 128).
# stride (int): An integer or tuple representing the stride of the sliding window. Default is set to 32.
#
# Output :
#  A list of lists, where each inner list contains information about a patch
def slide_windows(name, shape, output_size=128, stride=32):
    # Start a nested MLflow run for this function
    with mlflow.start_run(nested=True, run_name="slide_windows"):
        output_size = (output_size, output_size)
        strides = (stride, stride)
        
        # Log parameters
        mlflow.log_param("slide_window_output_size", output_size)
        mlflow.log_param("slide_window_stride", stride)

        patches_list = []
        idx = 0
        while idx * strides[0] + output_size[0] <= shape[0]:
            top = idx * strides[0]
            j = 0
            while j * strides[1] + output_size[1] <= shape[1]:
                left = j * strides[1]
                patches_list.append([name, top, left, output_size[0], output_size[1]])
                j += 1

            if j * strides[1] < shape[1]:
                left = shape[1] - output_size[1]
                patches_list.append([name, top, left, output_size[0], output_size[1]])
            idx += 1

        if idx * strides[0] < shape[0]:
            top = shape[0] - output_size[0]
            j = 0
            while j * strides[1] + output_size[1] <= shape[1]:
                left = j * strides[1]
                patches_list.append([name, top, left, output_size[0], output_size[1]])
                j += 1

            if j * strides[1] < shape[1]:
                left = shape[1] - output_size[1]
                patches_list.append([name, top, left, output_size[0], output_size[1]])
        
        # Log the number of patches generated
        mlflow.log_metric("num_patches_generated", len(patches_list))        
                
        return patches_list


#get_patches generate patches from a collection of images
# Input :
# data: A dictionary where keys represent image names, and values are the corresponding images (arrays).
# stride: An integer representing the stride of the sliding window. Default is set to 32.
# output_size: An integer representing the size of the output patches. Default is set to 256.
#
# Output :
#  A list of lists, where each inner list contains information about a patch
def get_patches(data, stride: int = 32, output_size: int = 256):
    # Start a nested MLflow run for this function
    with mlflow.start_run(nested=True, run_name="get_patches"):
        # Log parameters
        mlflow.log_param("get_patches_stride", stride)
        mlflow.log_param("get_patches_output_size", output_size)
        
        patches = []
        # Iterate for each image
        for name in (data.keys()):
            shape = data[name].shape
            patches += slide_windows(name, shape, output_size=output_size, stride=stride)
        shuffle(patches)
            
        # Log the number of patches generated
        mlflow.log_metric("num_patches_generated", len(patches))        
        
        return patches


#generate_ratios calculates ratios for each patch based on the pixel values in the corresponding labeled masks
# Input :
# patches: A list of lists, where each inner list contains information about a patch (name, top, left, height, width)
# class_num: An integer representing the number of classes in the labeled masks. Default is set to 3.
#
# Output:
# A list of lists, where each inner list contains class ratios for a patch.
def generate_ratios(patches, class_num=3):
    # Start a nested MLflow run for this function
    with mlflow.start_run(nested=True, run_name="generate_ratios"):
        # Log parameter
        mlflow.log_param("generate_ratios_class_num", class_num)
        
        # initialize ratios to 0
        ratios = []
        for i in range(len(patches)):
            mask = plt.imread(new_label_dir + "/" + patches[0][i])[patches[1][i]: patches[1][i] + patches[3][i], patches[2][i]: patches[2][i] + patches[4][i]]
            mask = adjustMask(mask, class_num)
            size = 1.0
            for idx in range(len(mask.shape)):
                size *= mask.shape[idx]

            # Get ratio for the patch
            ratio = []
            for idx in range(class_num):
                ratio.append(np.sum(mask == idx) / size) 

            ratios.append(ratio)
        
        # Log the number of ratios generated
        mlflow.log_metric("num_ratios_generated", len(ratios))    
        
        return ratios


# UNet-parts

In [14]:
class DomainEnrich(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.domain_enrich = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.domain_enrich(x)
    
def conv3x3(in_planes, out_planes, stride=1, groups=1, dilation=1):
    """3x3 convolution with padding"""
    return nn.Conv2d(in_planes, out_planes, kernel_size=3, stride=stride,
                     padding=dilation, groups=groups, bias=False, dilation=dilation)


def conv1x1(in_planes, out_planes, stride=1):
    """1x1 convolution"""
    return nn.Conv2d(in_planes, out_planes, kernel_size=1, stride=stride, bias=False)


class BasicBlock(nn.Module):
    expansion = 1

    def __init__(self, inplanes, planes, stride=1, downsample=None, groups=1,
                 base_width=64, dilation=1, norm_layer=None):
        super(BasicBlock, self).__init__()
        if norm_layer is None:
            norm_layer = nn.BatchNorm2d
        if groups != 1 or base_width != 64:
            raise ValueError('BasicBlock only supports groups=1 and base_width=64')
        if dilation > 1:
            raise NotImplementedError("Dilation > 1 not supported in BasicBlock")
        # Both self.conv1 and self.downsample layers downsample the input when stride != 1
        self.conv1 = conv3x3(inplanes, planes, stride)
        self.bn1 = norm_layer(planes)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = conv3x3(planes, planes)
        self.bn2 = norm_layer(planes)
        self.downsample = downsample
        self.stride = stride

    def forward(self, x):
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        if self.downsample is not None:
            identity = self.downsample(x)

        out += identity
        out = self.relu(out)

        return out
    
    
class DomainEnrich_Block(nn.Module):
    def __init__(self, n_channels, n_classes):
        super().__init__()
        self.basic_block1 = BasicBlock(n_channels,n_classes)
        self.basic_block2 = BasicBlock(n_classes,n_classes)

    def forward(self, x):
        x = self.basic_block1(x)
        x = self.basic_block2(x)
        return x

class DomainEnrichLoss():
    def __init__(self):
        self.alpha = torch.tensor(1e0, dtype=torch.float32)
        self.beta = torch.tensor(1e0, dtype=torch.float32)
        self.gamma = torch.tensor(1e0, dtype=torch.float32)
        self.sigma = torch.tensor(1e0, dtype=torch.float32)
        self.zeta = torch.tensor(1e0, dtype=torch.float32)
        self.lambda1 = 0.0001 * 0.0001
        self.lambda2 = 0.0001 * 0.0001

    def __call__(self, net, mask):
        # Squeeze the mask and add a new axis to match the shape of the feature maps
        idx_bone = (mask == 1)
        idx_dirt = (mask == 0)

        if idx_bone.sum() == 0 or idx_dirt.sum() == 0:
            return torch.tensor(0.0, requires_grad=True).to(mask.device)

        # Expand the mask dimensions to match the shape of net.x_rdn1 and net.x_rdn2
        idx_bone = idx_bone.unsqueeze(1).expand_as(net.x_rdn1)
        idx_dirt = idx_dirt.unsqueeze(1).expand_as(net.x_rdn1)

        rdn1_bone = net.x_rdn1[idx_bone].view(-1, net.x_rdn1.shape[1])
        rdn1_dirt = net.x_rdn1[idx_dirt].view(-1, net.x_rdn1.shape[1])
        rdn2_bone = net.x_rdn2[idx_bone].view(-1, net.x_rdn2.shape[1])
        rdn2_dirt = net.x_rdn2[idx_dirt].view(-1, net.x_rdn2.shape[1])
        
        # Retain gradients for intermediate tensors
        rdn1_bone.retain_grad()
        rdn1_dirt.retain_grad()
        rdn2_bone.retain_grad()
        rdn2_dirt.retain_grad()

        rdn1_bone_norm2 = torch.norm(rdn1_bone, p=2)
        rdn1_dirt_norm2 = torch.norm(rdn1_dirt, p=2)
        rdn2_bone_norm2 = torch.norm(rdn2_bone, p=2)
        rdn2_dirt_norm2 = torch.norm(rdn2_dirt, p=2)
        rdn2_dirt_norm1 = torch.norm(rdn2_dirt, p=1)

        LDF_bone = - (self.alpha * rdn1_bone_norm2**2) + (self.beta * rdn1_dirt_norm2**2)
        LDF_dirt = (self.gamma * rdn2_bone_norm2**2) - (self.sigma * rdn2_dirt_norm2**2) + (self.zeta * rdn2_dirt_norm1)

        return torch.sigmoid(self.lambda1 * LDF_bone + self.lambda2 * LDF_dirt)




class DoubleConv(nn.Module):
    """(convolution => [BN] => ReLU) * 2"""

    def __init__(self, in_channels, out_channels, dropout_rate):
        super().__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate),  # Add dropout here
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate),  # Add dropout here
        )

    def forward(self, x):
        return self.double_conv(x)


class Down(nn.Module):
    """Downscaling with maxpool then double conv"""

    def __init__(self, in_channels, out_channels, dropout_rate):
        super().__init__()
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels, dropout_rate)
        )

    def forward(self, x):
        return self.maxpool_conv(x)


class Up(nn.Module):
    """Upscaling then double conv"""

    def __init__(self, in_channels, out_channels, dropout_rate, bilinear=True):
        super().__init__()

        # if bilinear, use the normal convolutions to reduce the number of channels
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        else:
            self.up = nn.ConvTranspose2d(in_channels // 2, in_channels // 2, kernel_size=2, stride=2)

        self.conv = DoubleConv(in_channels, out_channels, dropout_rate)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        # input is CHW
        diffY = torch.tensor([x2.size()[2] - x1.size()[2]])
        diffX = torch.tensor([x2.size()[3] - x1.size()[3]])

        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2,
                        diffY // 2, diffY - diffY // 2])
        # if you have padding issues, see
        # https://github.com/HaiyongJiang/U-Net-Pytorch-Unstructured-Buggy/commit/0e854509c2cea854e247a9c615f175f76fbb2e3a
        # https://github.com/xiaopeng-liao/Pytorch-UNet/commit/8ebac70e633bac59fc22bb5195e513d5832fb3bd
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)


class OutConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(OutConv, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        return self.conv(x)




# UNet Light RDN

In [15]:
class UNet_Light_RDN(nn.Module):
    def __init__(self, n_channels, n_classes, dropout_rate, bilinear=True):
        super(UNet_Light_RDN, self).__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes
        self.bilinear = bilinear

        self.rdn1 = DomainEnrich_Block(n_channels, 16)
        self.rdn2 = DomainEnrich_Block(n_channels, 16)

        self.inc = DoubleConv(32, 32, dropout_rate)
        
        #self.inc = DoubleConv(1, 32, dropout_rate)  # Pass dropout_rate   #only unet
        
        self.down1 = Down(32, 64, dropout_rate)  # Pass dropout_rate
        self.down2 = Down(64, 128, dropout_rate)  # Pass dropout_rate
        self.down3 = Down(128, 256, dropout_rate)  # Pass dropout_rate
        self.down4 = Down(256, 256, dropout_rate)  # Pass dropout_rate
        self.up1 = Up(512, 128, dropout_rate, bilinear)  # Pass dropout_rate
        self.up2 = Up(256, 64, dropout_rate, bilinear)  # Pass dropout_rate
        self.up3 = Up(128, 32, dropout_rate, bilinear)  # Pass dropout_rate
        self.up4 = Up(64, 32, dropout_rate, bilinear)  # Pass dropout_rate
        self.outc = OutConv(32, n_classes)

    def forward(self, x):
        self.x_rdn1 = self.rdn1(x)
        self.x_rdn2 = self.rdn2(x)
        x1 = self.inc(torch.cat((self.x_rdn2, self.x_rdn1), 1))
        
        #x1 = self.inc(x) #only unet
        
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        logits = self.outc(x)
        return logits


# Parameters

In [16]:
# Get list of files in the new directories
name_list = os.listdir(new_label_dir)

# Load images
label_list = {name: plt.imread(os.path.join(new_label_dir, name)) for name in name_list}
input_list = {name: plt.imread(os.path.join(new_input_dir, name)) for name in name_list}


#Divide each unsegmented picture into smaller picture as the training set
train_patches = pd.DataFrame(get_patches(input_list, stride=stride, output_size=output))
#Same thing for the labelised pictures as the label set
val_patches = pd.DataFrame(get_patches(label_list,   stride=stride, output_size=output))
# Print the structure of val_patches to debug
print(val_patches.head())

#Calculate the % of air, bones and dirt for each patch of the training set
ratios = pd.DataFrame(generate_ratios(val_patches, class_num=class_num))#, batch_size=100))


# Log the number of patches generated
mlflow.log_metric("num_train_patches", len(train_patches))
mlflow.log_metric("num_val_patches", len(val_patches))

# Log the mean ratios for each class
mean_ratios = ratios.mean(axis=0)
mlflow.log_metric("mean_air_ratio", mean_ratios[0])
mlflow.log_metric("mean_bone_ratio", mean_ratios[1])
mlflow.log_metric("mean_dirt_ratio", mean_ratios[2])


                   0  1    2    3    4
0  ROU_2006_0800.tif  0    0  256  256
1  ROU_2006_0800.tif  0  128  256  256
2  ROU_2006_0800.tif  0  256  256  256
3  ROU_2006_0800.tif  0  384  256  256
4  ROU_2006_0800.tif  0  473  256  256


In [17]:
net = UNet_Light_RDN(n_channels=n_channel, n_classes=class_num, dropout_rate=dropout_rate)
net.to(device)

# Log the model
mlflow.pytorch.log_model(net, "model")

c:\Users\n.vanderesse\AppData\Local\ia-sereos_env\lib\site-packages\_distutils_hack\__init__.py:33: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")


In [18]:
optimizer = Adam(net.parameters(),
                      lr=float(learning_rate),
                      eps=float(epsilon),
                      betas=(0.9, 0.999),
                      weight_decay=weight_decay)

#learning rate schedule
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=period_size, gamma=0.1)

# Journalisation des hyperparamètres de l'optimiseur
mlflow.log_param("optimizer", "Adam")
mlflow.log_param("betas", (0.9, 0.999))


(0.9, 0.999)

In [19]:
# create train transform
train_transform = transforms.Compose([Augmentation(output_size=64), #config['output_size']
                                                    AdjustMask(class_num=class_num),
                                                    Normalize(max=255, min=0),
                                                    ToTensor()])

val_transform = transforms.Compose([AdjustMask(class_num=class_num),
                                                    Normalize(max=255, min=0),
                                                    ToTensor()])

# Training loop

In [20]:
# Initialize lists to collect metrics from nested runs
nested_metrics = {
    'train_loss': [],
    'val_accuracy': [],
    'dice_overlap_class_0': [],
    'dice_overlap_class_1': [],
    'dice_overlap_class_2': []
}

# Function to visualize images on TensorBoard
def visualize_images(writer, net, image, mask, device, nb_ite, last_batches, epoch):
    with torch.no_grad():
        pred2 = net(image)
    m2 = mask.argmax(1).cpu().numpy()
    pred2 = pred2.argmax(1).cpu().numpy()
    
    print("Shape of m2:", m2.shape)
    print("Shape of pred2:", pred2.shape)
    
    color_dict = [[0.0], [128.0 / 255.0], [1.0]]
    
    if len(m2.shape) == 2:  # If 2D, add a batch dimension
        m2 = np.expand_dims(m2, axis=0)
        pred2 = np.expand_dims(pred2, axis=0)
    
    batch_size, height, width = m2.shape  # Adjust this line if necessary after printing shapes
    pred_img = np.zeros((batch_size, 1, height, width), dtype=np.float32)
    mask_img = np.zeros((batch_size, 1, height, width), dtype=np.float32)
    
    for i in range(batch_size):  # Iterate over batch
        for j in range(height):  # Iterate over height
            for k in range(width):  # Iterate over width
                pred_img[i, 0, j, k] = color_dict[int(pred2[i, j, k])][0]
                mask_img[i, 0, j, k] = color_dict[int(m2[i, j, k])][0]
    
    writer.add_image('input_image', torchvision.utils.make_grid(image.cpu()), nb_ite + last_batches)
    writer.add_image('prediction_image', torchvision.utils.make_grid(torch.tensor(pred_img).to(device)), nb_ite + last_batches)
    writer.add_image('mask_image', torchvision.utils.make_grid(torch.tensor(mask_img).to(device)), nb_ite + last_batches)

# rdn_train train the model for one iteration
# Input : 
# net: The neural network model being trained.
# optimizer: The optimizer used for updating the model's parameters.
# data_loader: DataLoader providing batches of training data.
# epoch: Current epoch number (optional).
# total_epoch: Total number of epochs (optional).
# tensorboard_plot: Boolean flag for visualizing results on TensorBoard (default is False).
#
# Output :
# returns the total number of iterations processed for this epoch (nb_ite + last_batches)
def rdn_train(net, optimizer, data_loader, epoch=None, total_epoch=None, tensorboard_plot=False, nb_ite=0):
    max_batches1 = len(data_loader.dataset) // data_loader.batch_size + (1 if (len(data_loader.dataset) % data_loader.batch_size) != 0 else 0)

    # the epoch message for printing
    epoch_print = 'Epoch:'
    if epoch is not None:
        epoch_print += f'{epoch + 1}'
    if total_epoch is not None:
        epoch_print += f'/{total_epoch}'
    last_batches = 0.0
    loss1_sum = 0.0
    loss2_sum = 0.0
    ite = 0

    with tqdm(total=len(data_loader.dataset), desc=epoch_print, unit=' batches') as pbar:
        for i_batches, sample_batched in enumerate(data_loader):
            last_batches = i_batches
            mask = sample_batched['mask']
            image = sample_batched['image']
            # index = sample_batched['index']

            # convert to gpu
            mask = mask.to(device).long()
            image = image.to(device)

            # prediction
            pred = net(image)
            
            loss1 = DomainEnrichLoss()(net, mask)
            #loss1 = torch.Tensor(0)
            mask = create_one_hot(mask)
            
            CE_loss = nn.CrossEntropyLoss()
            loss2 = CE_loss(pred, mask)
            
            loss = loss2 + loss1
            
            loss1.to(device)
            loss2.to(device)
            loss.to(device)

            # Collecting train loss for the current batch
            nested_metrics['train_loss'].append(loss2.item())

            # backward
            optimizer.zero_grad()
            loss2.backward()
            #print("Gradient for loss1:", net.x_rdn1.grad)
            optimizer.step()
            
            if tensorboard_plot and epoch == 0 and ite == 0:
                writer.add_graph(net, image)
            if tensorboard_plot and (ite % (max_batches1 // 3) == 0):
                visualize_images(writer, net, image, mask, device, ite, last_batches, epoch)

            # Print results
            pbar.update(mask.shape[0])
            pbar.set_postfix(loss=loss.cpu().data.numpy(), loss1=loss1.cpu().data.numpy(), loss2=loss2.cpu().data.numpy())
            loss1_sum = loss1_sum + loss1.cpu().data.numpy()
            loss2_sum = loss2_sum + loss2.cpu().data.numpy()
            
            ite += 1
        avg_loss2 = loss2_sum / (last_batches + 1)
        avg_loss1 = loss1_sum / (last_batches + 1)
        avg_loss = avg_loss1 + avg_loss2
        writer.add_scalar('avg_train_loss2', avg_loss2)
        writer.add_scalar('avg_train_loss1', avg_loss1)
        writer.add_scalar('avg_train_loss', avg_loss)
        print(f'\nAverage, loss2: {(loss2_sum/ (last_batches + 1)):.6f}.')
        print(f'\nAverage, loss1: {(loss1_sum/ (last_batches + 1)):.6f}.')
        print(f'\nAverage, loss: {(avg_loss):.6f}.')
        
        
        # Log metrics to MLflow
        mlflow.log_metric("avg_train_loss2", avg_loss2, step=epoch)
        mlflow.log_metric("avg_train_loss1", avg_loss1, step=epoch)
        
    return nb_ite + last_batches

# rdn_val test the accuracy of the model for the current epoch
# Input :
# net: The neural network model.
# data_set: The dataset for validation.
# i_epoch: Current epoch number (optional).
# class_num: Number of classes in the dataset (default is 3).
#
# Output :
# returns the accuracy (criterion_value) and class-wise dice overlap results.
def rdn_val(net, data_set, i_epoch=None, class_num=3):
    dice_overlap = DiceOverlap(class_num)

    # check whether net is in train mode or not
    origin_is_train_mode = net.training

    # change the net to eval mode
    if origin_is_train_mode:
        net.eval()

    # check whether data set is in train mode
    data_set.val()

    criterion_value_sum = 0.0
    data_loader = DataLoader(data_set, batch_size=1, num_workers=0)
    dice_overlap_results = 0.0

    for i_batches, sample_batched in enumerate(data_loader):
        mask = sample_batched['mask']
        image = sample_batched['image']

        mask = mask.to(device)
        image = image.to(device)

        # prediction
        with torch.no_grad():
            if image.shape == (1, 1, 256, 256):
                pred = net(image)
                criterion_value_sum += Accuracy()(pred, mask.long()).cpu().data.numpy()

                if dice_overlap is not None:
                    dice_overlap_results += dice_overlap(pred, mask.long())

    criterion_value = criterion_value_sum / len(data_loader.dataset)
    dice_overlap_results = dice_overlap_results / len(data_loader.dataset)

    for i in range(dice_overlap_results.shape[0]):
        print(f'Class: {i:.0f}, Dice Overlap: {dice_overlap_results[i]:.6f}')

    if origin_is_train_mode:
        net.train()
        
    # Log metrics to MLflow
    mlflow.log_metric("val_accuracy", criterion_value, step=i_epoch)
    #writer.add_scalar("val_accuracy", criterion_value, step=i_epoch)
    for i in range(dice_overlap_results.shape[0]):
        mlflow.log_metric(f"dice_overlap_class_{i}", dice_overlap_results[i], step=i_epoch)
        # Collecting validation metrics
        nested_metrics['val_accuracy'].append(criterion_value)
        nested_metrics[f'dice_overlap_class_{i}'].append(dice_overlap_results[i])
    
    # print message
    if i_epoch is not None:
        print(f"Epoch: {i_epoch + 1}, Accuracy Value: {criterion_value:.6f}")
        writer.add_scalar('Accuracy Value', criterion_value, i_epoch)
        writer.add_scalars('Dice Overlap', {'Air': dice_overlap_results[0], 'Dirt': dice_overlap_results[1], 'Bone': dice_overlap_results[2]}, i_epoch)
    return criterion_value, dice_overlap_results


class DiceOverlap():
    def __init__(self, class_num):
        self.len = class_num

    def __call__(self, input, target):
        input = F.sigmoid(input)
        input = torch.max(input, 1)[1]

        dice = []

        for i in range(self.len):
            sub_target = torch.zeros(target.shape).cuda()
            sub_target[target == i] = 1
            sub_input = torch.zeros(input.shape).cuda()
            sub_input[input == i] = 1

            tp_idx = target == i

            eps = 0.0001
            tp = torch.sum(sub_input[tp_idx] == sub_target[tp_idx])
            fn = torch.sum(sub_input != sub_target)
            tp = tp.float()
            fn = fn.float()
            result = (2 * tp + eps) / (2 * tp + fn + eps)
            dice.append(result.cpu().data.numpy())

        return np.asarray(dice)
    
class Accuracy():
    def __call__(self, input, target, **kwargs):
        input = torch.max(input, 1)[1]
        size = 1
        for i in range(len(input.shape)):
            size = size * input.shape[i]
        return torch.sum(input == target).float() / size


In [21]:
# Training loop
epoch_count = 0
nb_ite = 0

print(f"Epoch progress:")
print(f"Progress training {epochs_num} epochs...")

total_timer = timer()

for i_epoch in range(epochs_num):
    with mlflow.start_run(nested=True, run_name=f"epoch_{i_epoch + 1}"):
        print(f"Epoch {epoch_count + 1} of {epochs_num}")

        if i_epoch < period_size:
            air_rate = 0.1
        elif i_epoch < 2 * period_size and i_epoch >= period_size:
            air_rate = 0.2
        elif i_epoch < 3 * period_size and i_epoch >= 2 * period_size:
            air_rate = 0.4
        else:
            air_rate = 0.5

        # Log parameter for the current epoch
        mlflow.log_param("air_rate", air_rate)

        # Get patches 
        patches = get_minimum_dirt_patches(dirt_choose_threshold=0.1, dirt_rate=0, patches=train_patches, ratios=ratios)
        DEB_patches, index = get_dirt_bone_patches(train_patches, ratios, air_rate)
        DEB_patches = pd.DataFrame(DEB_patches)

        data_set = HDF52D(patches, val_patches, train_transform=train_transform, val_transform=val_transform)
        DEB_data_set = HDF52D(DEB_patches, val_patches, train_transform=train_transform, val_transform=val_transform, train_idx=index)

        current_batch = int(batch_size)

        train_data_loader = DataLoader(dataset=DEB_data_set, batch_size=current_batch, shuffle=True, num_workers=0)

        print(f"learning rate {optimizer.param_groups[0]['lr']:.6f}")
        nb_ite = rdn_train(net, optimizer, train_data_loader, epoch=i_epoch, total_epoch=epochs_num, tensorboard_plot=True, nb_ite=nb_ite)

        # validating
        val_loss, class_val = rdn_val(net, data_set, i_epoch=i_epoch, class_num=class_num)

        # Collect validation metrics for the current epoch
        nested_metrics['val_accuracy'].append(val_loss)
        for i, class_val_metric in enumerate(class_val):
            nested_metrics[f'dice_overlap_class_{i}'].append(class_val_metric)

        class_val = pd.DataFrame(class_val)
        class_val.columns = ["Class Dice overlap"]
        print(class_val)

        epoch_count += 1
        iteration = np.floor((100 * epoch_count) / int(epochs_num))

# save model
torch.save(net.state_dict(), "RDN_DEB.pth")

# Aggregate and log metrics from nested runs to the main run
avg_train_loss = np.mean(nested_metrics['train_loss'])
avg_val_accuracy = np.mean(nested_metrics['val_accuracy'])
avg_dice_overlap_class_0 = np.mean(nested_metrics['dice_overlap_class_0'])
avg_dice_overlap_class_1 = np.mean(nested_metrics['dice_overlap_class_1'])
avg_dice_overlap_class_2 = np.mean(nested_metrics['dice_overlap_class_2'])

mlflow.log_metric("avg_train_loss", avg_train_loss)
mlflow.log_metric("avg_val_accuracy", avg_val_accuracy)
mlflow.log_metric("avg_dice_overlap_class_0", avg_dice_overlap_class_0)
mlflow.log_metric("avg_dice_overlap_class_1", avg_dice_overlap_class_1)
mlflow.log_metric("avg_dice_overlap_class_2", avg_dice_overlap_class_2)

# End MLflow run
mlflow.end_run()

# Close the TensorBoard writer
writer.close()


Epoch progress:
Progress training 10 epochs...
Epoch 1 of 10
There are 92 bone and 0 dirt patches in the training data...
learning rate 0.001000


Epoch:1/10:   0%|          | 0/92 [00:00<?, ? batches/s]C:\Users\n.vanderesse\AppData\Local\Temp\ipykernel_18476\2092351662.py:173: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  diffY = torch.tensor([x2.size()[2] - x1.size()[2]])
C:\Users\n.vanderesse\AppData\Local\Temp\ipykernel_18476\2092351662.py:174: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  diffX = torch.tensor([x2.size()[3] - x1.size()[3]])


Shape of m2: (32, 64, 64)
Shape of pred2: (32, 64, 64)


Epoch:1/10:  35%|███▍      | 32/92 [00:03<00:06,  9.90 batches/s, loss=1.5976038, loss1=0.5030326, loss2=1.0945711]

Shape of m2: (32, 64, 64)
Shape of pred2: (32, 64, 64)


...

Shape of m2: (32, 64, 64)
Shape of pred2: (32, 64, 64)


Epoch:10/10:  70%|██████▉   | 64/92 [00:00<00:00, 66.59 batches/s, loss=0.8801568, loss1=0.50280493, loss2=0.37735188]

Shape of m2: (28, 64, 64)
Shape of pred2: (28, 64, 64)


Epoch:10/10: 100%|██████████| 92/92 [00:01<00:00, 66.14 batches/s, loss=0.87821484, loss1=0.502534, loss2=0.37568086] 



Average, loss2: 0.380127.

Average, loss1: 0.502697.

Average, loss: 0.882824.
Class: 0, Dice Overlap: 0.992806
Class: 1, Dice Overlap: 0.565066
Class: 2, Dice Overlap: 0.923255
Epoch: 10, Accuracy Value: 0.983968
   Class Dice overlap
0            0.992806
1            0.565066
2            0.923255
